# Multi-Model Trading Architecture Training

This notebook trains the three-model ensemble for Solana memecoin trading:

1. **Model 1 (Screener)**: XGBoost classifier - determines if a token is worth monitoring
2. **Model 2 (Entry)**: LSTM/Hybrid - determines optimal entry timing
3. **Model 3 (Exit)**: XGBoost + Rules - determines optimal exit point

## Architecture Overview

```
Token Stream --> [SCREENER] --> [ENTRY] --> [EXIT] --> Trade Execution
                 (XGBoost)     (LSTM)      (Rules+ML)
```

## Requirements
- GPU recommended (A100, L4, T4)
- ~30-60 minutes total training time
- Preprocessed datasets included in repository

## 1. Setup & Clone Repository

In [ ]:
# GitHub Configuration
GITHUB_USERNAME = "tr4m0ryp"  # Your GitHub username
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # Your Personal Access Token
GITHUB_REPO_URL = "https://github.com/tr4m0ryp/GMGN_TradingBot.git"

print("GitHub credentials configured")
print(f"Username: {GITHUB_USERNAME}")
print(f"Repository: {GITHUB_REPO_URL}")

In [ ]:
import os
import subprocess

os.chdir('/content')

# Extract repo name from URL
REPO_NAME = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
REPO_PATH = f'/content/{REPO_NAME}'

if os.path.exists(REPO_PATH):
    print(f"Repository '{REPO_NAME}' already exists at {REPO_PATH}")
    print("Pulling latest changes...")
    os.chdir(REPO_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout if result.stdout else "Already up to date.")
else:
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')
    result = subprocess.run(['git', 'clone', clone_url], capture_output=True, text=True, check=True)
    print(f"Successfully cloned repository to {REPO_PATH}")

# Set paths
AI_MODEL_PATH = f'{REPO_PATH}/ai_model_v2'
DATA_PATH = f'{AI_MODEL_PATH}/data/processed'
OUTPUT_DIR = f'{AI_MODEL_PATH}/models/checkpoints'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(AI_MODEL_PATH)

print(f"\nWorking directory: {os.getcwd()}")
print(f"Data path: {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Install dependencies
!pip install -q torch numpy pandas xgboost scikit-learn matplotlib tqdm

import pickle
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print(f"[OK] PyTorch {torch.__version__}")
print(f"[OK] XGBoost {xgb.__version__}")
print(f"[OK] NumPy {np.__version__}")

In [ ]:
# Check GPU and set device
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE = 'cuda'
    torch.backends.cudnn.benchmark = True
    print(f"[GPU] {gpu_name} | {total_mem:.1f}GB | CUDA {torch.version.cuda}")
else:
    DEVICE = 'cpu'
    print("[WARN] No GPU available - training will be slower")

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\nDevice: {DEVICE}")
print(f"Random seed: {SEED}")

## 2. Load Preprocessed Datasets

In [ ]:
# Load dataset metadata
with open(f'{DATA_PATH}/metadata.json', 'r') as f:
    metadata = json.load(f)

print("=" * 60)
print("DATASET METADATA")
print("=" * 60)
print(f"Total tokens: {metadata['token_counts']['total']}")
print(f"Train/Val/Test split: {metadata['token_counts']['train']}/{metadata['token_counts']['val']}/{metadata['token_counts']['test']}")
print(f"\nScreener samples: {metadata['screener']}")
print(f"Entry samples: {metadata['entry']}")
print(f"Exit samples: {metadata['exit']}")
print("=" * 60)

In [ ]:
# Load screener data (Model 1)
with open(f'{DATA_PATH}/screener_train.pkl', 'rb') as f:
    screener_train = pickle.load(f)
with open(f'{DATA_PATH}/screener_val.pkl', 'rb') as f:
    screener_val = pickle.load(f)
with open(f'{DATA_PATH}/screener_test.pkl', 'rb') as f:
    screener_test = pickle.load(f)

print("[Model 1] Screener Data Loaded")
print(f"  Train: X={screener_train['X'].shape}, y={screener_train['y'].shape}")
print(f"  Val:   X={screener_val['X'].shape}, y={screener_val['y'].shape}")
print(f"  Test:  X={screener_test['X'].shape}, y={screener_test['y'].shape}")
print(f"  Features: {screener_train['feature_names'][:5]}...")
print(f"  Class balance (train): WORTHY={screener_train['y'].sum()}, AVOID={len(screener_train['y']) - screener_train['y'].sum()}")

In [ ]:
# Load entry data (Model 2)
with open(f'{DATA_PATH}/entry_train.pkl', 'rb') as f:
    entry_train = pickle.load(f)
with open(f'{DATA_PATH}/entry_val.pkl', 'rb') as f:
    entry_val = pickle.load(f)
with open(f'{DATA_PATH}/entry_test.pkl', 'rb') as f:
    entry_test = pickle.load(f)

print("[Model 2] Entry Data Loaded")
print(f"  Train: {len(entry_train['sequences'])} sequences")
print(f"  Val:   {len(entry_val['sequences'])} sequences")
print(f"  Test:  {len(entry_test['sequences'])} sequences")
print(f"  Sample sequence shape: {entry_train['sequences'][0].shape}")
print(f"  Label distribution (train): {np.bincount(entry_train['labels'])}")
print(f"    (0=WAIT, 1=ENTER_NOW, 2=ABORT)")

In [ ]:
# Load exit data (Model 3)
with open(f'{DATA_PATH}/exit_train.pkl', 'rb') as f:
    exit_train = pickle.load(f)
with open(f'{DATA_PATH}/exit_val.pkl', 'rb') as f:
    exit_val = pickle.load(f)
with open(f'{DATA_PATH}/exit_test.pkl', 'rb') as f:
    exit_test = pickle.load(f)

print("[Model 3] Exit Data Loaded")
print(f"  Train: X={exit_train['X'].shape}, y={exit_train['y'].shape}")
print(f"  Val:   X={exit_val['X'].shape}, y={exit_val['y'].shape}")
print(f"  Test:  X={exit_test['X'].shape}, y={exit_test['y'].shape}")
print(f"  Label distribution (train): {np.bincount(exit_train['y'])}")
print(f"    (0=HOLD, 1=EXIT_NOW, 2=PARTIAL_EXIT)")

## 3. Train Model 1: Screener (XGBoost)

In [ ]:
# Screener configuration
SCREENER_CONFIG = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
    'device': 'cuda' if DEVICE == 'cuda' else 'cpu',
}

# Calculate scale_pos_weight for class imbalance
n_neg = (screener_train['y'] == 0).sum()
n_pos = (screener_train['y'] == 1).sum()
SCREENER_CONFIG['scale_pos_weight'] = n_neg / n_pos

print("Screener Configuration:")
print(f"  n_estimators: {SCREENER_CONFIG['n_estimators']}")
print(f"  max_depth: {SCREENER_CONFIG['max_depth']}")
print(f"  scale_pos_weight: {SCREENER_CONFIG['scale_pos_weight']:.2f}")

In [ ]:
print("Training Model 1: Entry Worthiness Screener...\n")

# Create XGBoost classifier
screener_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric=['logloss', 'auc'],
    early_stopping_rounds=20,
    **SCREENER_CONFIG
)

# Train with validation
screener_model.fit(
    screener_train['X'], screener_train['y'],
    eval_set=[(screener_val['X'], screener_val['y'])],
    verbose=True
)

print("\nScreener training complete!")

In [ ]:
# Evaluate screener on test set
screener_pred = screener_model.predict(screener_test['X'])
screener_pred_proba = screener_model.predict_proba(screener_test['X'])[:, 1]

screener_metrics = {
    'accuracy': accuracy_score(screener_test['y'], screener_pred),
    'precision': precision_score(screener_test['y'], screener_pred, zero_division=0),
    'recall': recall_score(screener_test['y'], screener_pred, zero_division=0),
    'f1': f1_score(screener_test['y'], screener_pred, zero_division=0),
    'roc_auc': roc_auc_score(screener_test['y'], screener_pred_proba),
}

print("=" * 50)
print("MODEL 1: SCREENER TEST METRICS")
print("=" * 50)
for name, value in screener_metrics.items():
    print(f"  {name:12s}: {value:.4f}")
print("=" * 50)

In [ ]:
# Feature importance
importance = screener_model.feature_importances_
feature_names = screener_train['feature_names']
sorted_idx = np.argsort(importance)[::-1]

print("Top 10 Feature Importances:")
for i in range(min(10, len(sorted_idx))):
    idx = sorted_idx[i]
    print(f"  {i+1}. {feature_names[idx]:25s} {importance[idx]:.4f}")

## 4. Train Model 2: Entry Timing (LSTM)

In [ ]:
# Entry model configuration
ENTRY_CONFIG = {
    'input_dim': entry_train['sequences'][0].shape[1],  # 14 features
    'hidden_dim': 128,
    'num_layers': 2,
    'num_classes': 3,  # WAIT, ENTER_NOW, ABORT
    'dropout': 0.3,
    'bidirectional': True,
    'batch_size': 64,
    'epochs': 50,
    'patience': 10,
    'learning_rate': 0.001,
}

print("Entry Model Configuration:")
for key, value in ENTRY_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Define LSTM model with attention
class EntryLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3, bidirectional=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * self.num_directions, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * self.num_directions, num_classes)
    
    def forward(self, x, lengths=None):
        # x: (batch, seq_len, input_dim)
        lstm_out, _ = self.lstm(x)
        # lstm_out: (batch, seq_len, hidden_dim * num_directions)
        
        # Attention weights
        attn_weights = self.attention(lstm_out)  # (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        
        # Weighted sum
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (batch, hidden_dim * num_directions)
        
        out = self.dropout(context)
        out = self.fc(out)
        return out

print("EntryLSTM model defined.")

In [ ]:
# Create PyTorch datasets
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.LongTensor([self.labels[idx]])[0]

def collate_fn(batch):
    sequences, labels = zip(*batch)
    # Pad sequences to max length in batch
    max_len = max(s.shape[0] for s in sequences)
    padded = torch.zeros(len(sequences), max_len, sequences[0].shape[1])
    lengths = []
    for i, s in enumerate(sequences):
        padded[i, :s.shape[0], :] = s
        lengths.append(s.shape[0])
    return padded, torch.stack(labels), torch.LongTensor(lengths)

# Create data loaders
train_dataset = SequenceDataset(entry_train['sequences'], entry_train['labels'])
val_dataset = SequenceDataset(entry_val['sequences'], entry_val['labels'])
test_dataset = SequenceDataset(entry_test['sequences'], entry_test['labels'])

train_loader = DataLoader(train_dataset, batch_size=ENTRY_CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=ENTRY_CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=ENTRY_CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
print("Training Model 2: Entry Timing Optimizer...\n")

# Initialize model
entry_model = EntryLSTM(
    input_dim=ENTRY_CONFIG['input_dim'],
    hidden_dim=ENTRY_CONFIG['hidden_dim'],
    num_layers=ENTRY_CONFIG['num_layers'],
    num_classes=ENTRY_CONFIG['num_classes'],
    dropout=ENTRY_CONFIG['dropout'],
    bidirectional=ENTRY_CONFIG['bidirectional'],
).to(DEVICE)

# Calculate class weights for imbalanced data
class_counts = np.bincount(entry_train['labels'])
class_weights = torch.FloatTensor(len(class_counts) / (len(class_counts) * class_counts)).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(entry_model.parameters(), lr=ENTRY_CONFIG['learning_rate'], weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"Model parameters: {sum(p.numel() for p in entry_model.parameters()):,}")
print(f"Class weights: {class_weights.cpu().numpy()}")

In [ ]:
# Training loop
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

for epoch in range(ENTRY_CONFIG['epochs']):
    # Training
    entry_model.train()
    train_loss = 0.0
    for batch_x, batch_y, lengths in tqdm(train_loader, desc=f"Epoch {epoch+1}/{ENTRY_CONFIG['epochs']}", leave=False):
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = entry_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(entry_model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    entry_model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y, lengths in val_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            outputs = entry_model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(batch_y.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    val_acc = accuracy_score(val_labels, val_preds)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(entry_model.state_dict(), f'{OUTPUT_DIR}/entry_model_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= ENTRY_CONFIG['patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Load best model
entry_model.load_state_dict(torch.load(f'{OUTPUT_DIR}/entry_model_best.pt'))
print("\nEntry model training complete!")

In [ ]:
# Evaluate entry model on test set
entry_model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for batch_x, batch_y, lengths in test_loader:
        batch_x = batch_x.to(DEVICE)
        outputs = entry_model(batch_x)
        preds = torch.argmax(outputs, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(batch_y.numpy())

entry_metrics = {
    'accuracy': accuracy_score(test_labels, test_preds),
    'precision_macro': precision_score(test_labels, test_preds, average='macro', zero_division=0),
    'recall_macro': recall_score(test_labels, test_preds, average='macro', zero_division=0),
    'f1_macro': f1_score(test_labels, test_preds, average='macro', zero_division=0),
}

print("=" * 50)
print("MODEL 2: ENTRY TEST METRICS")
print("=" * 50)
for name, value in entry_metrics.items():
    print(f"  {name:16s}: {value:.4f}")
print("=" * 50)

## 5. Train Model 3: Exit Optimizer (XGBoost)

In [ ]:
# Exit model configuration
EXIT_CONFIG = {
    'n_estimators': 150,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist',
    'device': 'cuda' if DEVICE == 'cuda' else 'cpu',
}

print("Exit Model Configuration:")
print(f"  n_estimators: {EXIT_CONFIG['n_estimators']}")
print(f"  max_depth: {EXIT_CONFIG['max_depth']}")

In [ ]:
print("Training Model 3: Exit Point Optimizer...\n")

# Create XGBoost classifier for multi-class
exit_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    **EXIT_CONFIG
)

# Train with validation
exit_model.fit(
    exit_train['X'], exit_train['y'],
    eval_set=[(exit_val['X'], exit_val['y'])],
    verbose=True
)

print("\nExit model training complete!")

In [ ]:
# Evaluate exit model on test set
exit_pred = exit_model.predict(exit_test['X'])

exit_metrics = {
    'accuracy': accuracy_score(exit_test['y'], exit_pred),
    'precision_macro': precision_score(exit_test['y'], exit_pred, average='macro', zero_division=0),
    'recall_macro': recall_score(exit_test['y'], exit_pred, average='macro', zero_division=0),
    'f1_macro': f1_score(exit_test['y'], exit_pred, average='macro', zero_division=0),
}

print("=" * 50)
print("MODEL 3: EXIT TEST METRICS")
print("=" * 50)
for name, value in exit_metrics.items():
    print(f"  {name:16s}: {value:.4f}")
print("=" * 50)

## 6. Save Models Locally

In [ ]:
import shutil

# Save all models
print("Saving models...\n")

# Save screener model
screener_model.save_model(f'{OUTPUT_DIR}/screener_model.json')
print(f"[SAVED] screener_model.json")

# Save entry model (already saved during training, but save final version)
torch.save({
    'model_state_dict': entry_model.state_dict(),
    'config': ENTRY_CONFIG,
}, f'{OUTPUT_DIR}/entry_model.pt')
print(f"[SAVED] entry_model.pt")

# Save exit model
exit_model.save_model(f'{OUTPUT_DIR}/exit_model.json')
print(f"[SAVED] exit_model.json")

# Save training metrics
training_summary = {
    'screener_metrics': screener_metrics,
    'entry_metrics': entry_metrics,
    'exit_metrics': exit_metrics,
    'screener_config': SCREENER_CONFIG,
    'entry_config': ENTRY_CONFIG,
    'exit_config': EXIT_CONFIG,
}

with open(f'{OUTPUT_DIR}/training_summary.json', 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    def convert(obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, dict):
            return {k: convert(v) for k, v in obj.items()}
        return obj
    json.dump(convert(training_summary), f, indent=2)
print(f"[SAVED] training_summary.json")

print(f"\nAll models saved to: {OUTPUT_DIR}")

In [ ]:
# List saved files
print("Saved files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:30s} {size_kb:8.1f} KB")

In [ ]:
# Create a zip file for easy download
ZIP_NAME = 'trained_models'
shutil.make_archive(f'/content/{ZIP_NAME}', 'zip', OUTPUT_DIR)
print(f"Created: /content/{ZIP_NAME}.zip")

# Download the zip file
from google.colab import files
files.download(f'/content/{ZIP_NAME}.zip')
print("\nDownload started! Check your browser downloads.")

## 7. Training Summary

In [ ]:
print("\n" + "=" * 70)
print("                   TRAINING COMPLETE")
print("=" * 70)

print(f"\n[MODEL 1] Screener (XGBoost)")
print(f"  Accuracy:  {screener_metrics['accuracy']:.4f}")
print(f"  ROC-AUC:   {screener_metrics['roc_auc']:.4f}")
print(f"  Precision: {screener_metrics['precision']:.4f}")
print(f"  Recall:    {screener_metrics['recall']:.4f}")

print(f"\n[MODEL 2] Entry Timing (LSTM)")
print(f"  Accuracy:  {entry_metrics['accuracy']:.4f}")
print(f"  F1 (macro): {entry_metrics['f1_macro']:.4f}")

print(f"\n[MODEL 3] Exit Optimizer (XGBoost)")
print(f"  Accuracy:  {exit_metrics['accuracy']:.4f}")
print(f"  F1 (macro): {exit_metrics['f1_macro']:.4f}")

print(f"\n[OUTPUT] {OUTPUT_DIR}")
print(f"[DOWNLOAD] /content/{ZIP_NAME}.zip")
print("=" * 70)